In [27]:
import numpy as np
from scipy.linalg import eigh, svd
from scipy.sparse.linalg import eigsh, svds
import torch

In [2]:
import pickle

with open("Corr_len_cytnx.pkl", "rb") as file:
    loaded_data = pickle.load(file)
    
Corr_len_cytnx = loaded_data['Corr_len']
Corr_len_2_cytnx = loaded_data['Corr_len_2']
Corr_len_4_cytnx = loaded_data['Corr_len_4']
Corr_len_10_cytnx = loaded_data['Corr_len_10']

In [18]:
OPTIMIZE = False

def get_W(beta, J=1, h=0):
    """
    Calculate the matrix W based on beta, J, and h using numpy.
    
    Parameters:
        beta (float): Inverse temperature.
        J (float): Interaction strength (default is 1).
        h (float): External field strength (default is 0).

    Returns:
        np.ndarray: Matrix W.
    """
    sq_cosh = np.sqrt(np.cosh(beta * J))
    sq_sinh = np.sqrt(np.sinh(beta * J))
    W = np.array([
        [sq_cosh, sq_sinh],
        [sq_cosh, -sq_sinh]
    ])
    return W


def get_Wh(beta, J, h):
    """
    Calculate the matrix Wh incorporating the external field h using numpy.
    
    Parameters:
        beta (float): Inverse temperature.
        J (float): Interaction strength.
        h (float): External field strength.

    Returns:
        np.ndarray: Matrix Wh.
    """
    cosh = np.cosh(beta * J)
    sinh = np.sinh(beta * J)

    Wh = np.array([
        [np.sqrt(cosh) * np.exp(beta * h / 2), np.sqrt(sinh) * np.exp(beta * h / 2)],
        [np.sqrt(cosh) * np.exp(-beta * h / 2), -np.sqrt(sinh) * np.exp(-beta * h / 2)]
    ])
    return Wh


def get_T_bare(beta, J=1, h=0, get_W=get_W):
    """
    Calculate the bare transfer matrix T_bare using numpy.
    
    Parameters:
        beta (float): Inverse temperature.
        J (float): Interaction strength (default is 1).
        h (float): External field strength (default is 0).
        get_W (callable): Function to calculate W or Wh (default is get_W).

    Returns:
        np.ndarray: Bare transfer matrix T_bare.
    """
    W = get_W(beta, J=J, h=h)
    T_bare = np.einsum('aA,aB,aC,aD->ABCD', W, W, W, W)
    return T_bare


def merge_y(Tup, Tdn, combine=False, trace=False):
    """
    Merge tensors Tup and Tdn along the y-axis using numpy.

    Parameters:
        Tup (np.ndarray): Upper tensor.
        Tdn (np.ndarray): Lower tensor.
        combine (bool): Whether to combine dimensions.
        trace (bool): Whether to perform trace contraction.

    Returns:
        np.ndarray: Merged tensor.
    """
    if trace:
        Tmerge = np.einsum('bBDa,aCEb->BCDE', Tup, Tdn, optimize = OPTIMIZE)
        
        if combine:
            shape = Tmerge.shape
            Tmerge = Tmerge.reshape(shape[0] * shape[1], shape[2] * shape[3])
    else:
        Tmerge = np.einsum('ABDa,aCEF->ABCDEF', Tup, Tdn, optimize = OPTIMIZE)
        
        if combine:
            shape = Tmerge.shape
            Tmerge = Tmerge.reshape(shape[0], shape[1] * shape[2], shape[3] * shape[4], shape[5])
            
    return Tmerge

def merge_y_truncate(Tup, Tdn, dcut, niter=10):
    if ((Tup.shape[1] * Tdn.shape[1]) <= dcut):
        return merge_y(Tup, Tdn, True)
    
    # Compute the merged tensor
    Tmerge_pure = np.einsum('aAbc,cBef,aCbd,dDef->ABCD', Tup, Tdn, Tup, Tdn, optimize = OPTIMIZE)
    Tmerge_pure_shape = Tmerge_pure.shape
    Tmerge_pure = Tmerge_pure.reshape(
        Tmerge_pure_shape[0] * Tmerge_pure_shape[1], Tmerge_pure_shape[2] * Tmerge_pure_shape[3]
    )
    
    # Perform SVD with rank truncation
    U, S, Vt = svds(Tmerge_pure, k=dcut)
    U = U.reshape(Tmerge_pure_shape[0], Tmerge_pure_shape[1], dcut)
    
    # Reconstruct the truncated tensor
    Tmerge = np.einsum('Aace,ebdD,cdC,abB->ABCD', Tup, Tdn, U, U, optimize = OPTIMIZE)
    
    return Tmerge

def merge_x(TL, TR, combine=False, trace=False):
    if trace:
        I = np.identity(TL.shape[1])
        Tmerge = np.einsum('aAcb,ecBd,ab,ed->AB', TL, TR, I, I, optimize = OPTIMIZE)
    else:
        Tmerge = np.einsum('BCaF,AaDE->ABCDEF', TL, TR, optimize = OPTIMIZE)
        
        if combine:
            shape = Tmerge.shape
            Tmerge = Tmerge.reshape(shape[0] * shape[1], shape[2], shape[3], shape[4] * shape[5])
            
    return Tmerge


def merge_x(TL, TR, combine=False, trace=False):
    if trace:
        Tmerge = np.einsum('aAca,ecBe->AB', TL, TR, optimize = OPTIMIZE)
    else:
        Tmerge = np.einsum('BCaF,AaDE->ABCDEF', TL, TR, optimize = OPTIMIZE)
        
        if combine:
            shape = Tmerge.shape
            Tmerge = Tmerge.reshape(shape[0] * shape[1], shape[2], shape[3], shape[4] * shape[5])
            
    return Tmerge

def merge_x_truncate(TL, TR, dcut, niter=10):
    if ((TL.shape[0] * TR.shape[0]) <= dcut):
        return merge_x(TL, TR, True)
    
    # Compute the merged tensor
    Tmerge_pure = np.einsum('abcd,ecfg,hbid,jifg->aehj', TL, TR, TL, TR, optimize = OPTIMIZE)
    Tmerge_pure_shape = Tmerge_pure.shape
    Tmerge_pure = Tmerge_pure.reshape(
        Tmerge_pure_shape[0] * Tmerge_pure_shape[1], Tmerge_pure_shape[2] * Tmerge_pure_shape[3]
    )
    # print(Tmerge_pure.shape)
    # Perform SVD with rank truncation
    U, S, Vt = svds(Tmerge_pure, k=dcut)
    U = U.reshape(Tmerge_pure_shape[0], Tmerge_pure_shape[1], dcut)
    
    # Reconstruct the truncated tensor
    Tmerge = np.einsum('abcd,ecfg,aeh,dgi->hbfi', TL, TR, U, U, optimize = OPTIMIZE)
    return Tmerge


In [4]:
MaxL = 4
Temp = np.linspace(2.26,4,100)
W = np.ones((MaxL+1,len(Temp),2))
E = np.ones((MaxL+1,len(Temp),2))
Tc = 2/np.log(1+np.sqrt(2))

In [5]:
%%time

for i, temp in enumerate(Temp):
    T_bare = get_T_bare(1 / temp)
    TL = T_bare
    for j in range(2, MaxL + 1):
        TL_Trace = merge_y(TL, TL, combine=True, trace=True)
        eigvals, eigvecs = eigsh(TL_Trace, k=2, which='LM', return_eigenvectors=True)
        W[j, i, :] = eigvals
        del TL_Trace
        
        if j != MaxL:
            TL = merge_y(TL, TL, combine=True)
            
E = -np.log(W)
Corr_len = 1 / (E[:, :, -2] - E[:, :, -1])
np.allclose(Corr_len, Corr_len_cytnx)

CPU times: user 1.09 s, sys: 65.7 ms, total: 1.16 s
Wall time: 153 ms


<timed exec>:14: RuntimeWarning: divide by zero encountered in divide


True

In [6]:
%%time
dcut = 2

for i,temp in enumerate(Temp):
    T_bare = get_T_bare(1/temp)
    TL = T_bare
    for j in range(2,MaxL+1):
        TLx = merge_x_truncate(TL,TL,dcut)
        TL_Trace = merge_y(TLx,TLx,True,True)
        eigvals, eigvecs = eigsh(TL_Trace, k=2, which='LM', return_eigenvectors=True)
        W[j,i,:] = eigvals
        TL = merge_y_truncate(TLx,TLx,dcut)
        TL = TL/np.mean(np.abs(TL))
E = -np.log(W)
Corr_len_2 = 1/(E[:,:,-2]-E[:,:,-1])
np.allclose(Corr_len_2, Corr_len_2_cytnx)

CPU times: user 89.8 ms, sys: 997 μs, total: 90.8 ms
Wall time: 90.8 ms


<timed exec>:14: RuntimeWarning: divide by zero encountered in divide


True

In [19]:
%%time
dcut = 4

for i,temp in enumerate(Temp):
    T_bare = get_T_bare(1/temp)
    TL = T_bare
    for j in range(2,MaxL+1):
        TLx = merge_x_truncate(TL,TL,dcut)
        TL_Trace = merge_y(TLx,TLx,True,True)
        eigvals, eigvecs = eigsh(TL_Trace, k=2, which='LM', return_eigenvectors=True)
        W[j,i,:] = eigvals
        TL = merge_y_truncate(TLx,TLx,dcut)
        TL = TL/np.mean(np.abs(TL))
E = -np.log(W)
Corr_len_4 = 1/(E[:,:,-2]-E[:,:,-1])
np.allclose(Corr_len_4, Corr_len_4_cytnx)

CPU times: user 2.29 s, sys: 1 μs, total: 2.29 s
Wall time: 2.29 s


<timed exec>:14: RuntimeWarning: divide by zero encountered in divide


True

In [40]:
OPTIMIZE = True

def get_W(beta, J=1, h=0):
    """
    Calculate the matrix W based on beta, J, and h using numpy.
    
    Parameters:
        beta (float): Inverse temperature.
        J (float): Interaction strength (default is 1).
        h (float): External field strength (default is 0).

    Returns:
        np.ndarray: Matrix W.
    """
    sq_cosh = np.sqrt(np.cosh(beta * J))
    sq_sinh = np.sqrt(np.sinh(beta * J))
    W = np.array([
        [sq_cosh, sq_sinh],
        [sq_cosh, -sq_sinh]
    ])
    return W


def get_Wh(beta, J, h):
    """
    Calculate the matrix Wh incorporating the external field h using numpy.
    
    Parameters:
        beta (float): Inverse temperature.
        J (float): Interaction strength.
        h (float): External field strength.

    Returns:
        np.ndarray: Matrix Wh.
    """
    cosh = np.cosh(beta * J)
    sinh = np.sinh(beta * J)

    Wh = np.array([
        [np.sqrt(cosh) * np.exp(beta * h / 2), np.sqrt(sinh) * np.exp(beta * h / 2)],
        [np.sqrt(cosh) * np.exp(-beta * h / 2), -np.sqrt(sinh) * np.exp(-beta * h / 2)]
    ])
    return Wh


def get_T_bare(beta, J=1, h=0, get_W=get_W):
    """
    Calculate the bare transfer matrix T_bare using numpy.
    
    Parameters:
        beta (float): Inverse temperature.
        J (float): Interaction strength (default is 1).
        h (float): External field strength (default is 0).
        get_W (callable): Function to calculate W or Wh (default is get_W).

    Returns:
        np.ndarray: Bare transfer matrix T_bare.
    """
    W = get_W(beta, J=J, h=h)
    T_bare = np.einsum('aA,aB,aC,aD->ABCD', W, W, W, W)
    return T_bare


def merge_y(Tup, Tdn, combine=False, trace=False):
    """
    Merge tensors Tup and Tdn along the y-axis using numpy.

    Parameters:
        Tup (np.ndarray): Upper tensor.
        Tdn (np.ndarray): Lower tensor.
        combine (bool): Whether to combine dimensions.
        trace (bool): Whether to perform trace contraction.

    Returns:
        np.ndarray: Merged tensor.
    """
    if trace:
        Tmerge = np.einsum('bBDa,aCEb->BCDE', Tup, Tdn, optimize = OPTIMIZE)
        
        if combine:
            shape = Tmerge.shape
            Tmerge = Tmerge.reshape(shape[0] * shape[1], shape[2] * shape[3])
    else:
        Tmerge = np.einsum('ABDa,aCEF->ABCDEF', Tup, Tdn, optimize = OPTIMIZE)
        
        if combine:
            shape = Tmerge.shape
            Tmerge = Tmerge.reshape(shape[0], shape[1] * shape[2], shape[3] * shape[4], shape[5])
            
    return Tmerge

def merge_y_truncate(Tup, Tdn, dcut, niter=10):
    if ((Tup.shape[1] * Tdn.shape[1]) <= dcut):
        return merge_y(Tup, Tdn, True)
    
    # Compute the merged tensor
    Tmerge_pure = np.einsum('aAbc,cBef,aCbd,dDef->ABCD', Tup, Tdn, Tup, Tdn, optimize = OPTIMIZE)
    Tmerge_pure_shape = Tmerge_pure.shape
    Tmerge_pure = Tmerge_pure.reshape(
        Tmerge_pure_shape[0] * Tmerge_pure_shape[1], Tmerge_pure_shape[2] * Tmerge_pure_shape[3]
    )
    
    # Perform SVD with rank truncation
    U, S, Vt = torch.svd(torch.tensor(Tmerge_pure))
    U = U[:,:dcut].numpy()
    U = U.reshape(Tmerge_pure_shape[0], Tmerge_pure_shape[1], dcut)
    
    # Reconstruct the truncated tensor
    Tmerge = np.einsum('Aace,ebdD,cdC,abB->ABCD', Tup, Tdn, U, U, optimize = OPTIMIZE)
    
    return Tmerge

def merge_x(TL, TR, combine=False, trace=False):
    if trace:
        I = np.identity(TL.shape[1])
        Tmerge = np.einsum('aAcb,ecBd,ab,ed->AB', TL, TR, I, I, optimize = OPTIMIZE)
    else:
        Tmerge = np.einsum('BCaF,AaDE->ABCDEF', TL, TR, optimize = OPTIMIZE)
        
        if combine:
            shape = Tmerge.shape
            Tmerge = Tmerge.reshape(shape[0] * shape[1], shape[2], shape[3], shape[4] * shape[5])
            
    return Tmerge


def merge_x(TL, TR, combine=False, trace=False):
    if trace:
        Tmerge = np.einsum('aAca,ecBe->AB', TL, TR, optimize = OPTIMIZE)
    else:
        Tmerge = np.einsum('BCaF,AaDE->ABCDEF', TL, TR, optimize = OPTIMIZE)
        
        if combine:
            shape = Tmerge.shape
            Tmerge = Tmerge.reshape(shape[0] * shape[1], shape[2], shape[3], shape[4] * shape[5])
            
    return Tmerge

def merge_x_truncate(TL, TR, dcut, niter=10):
    if ((TL.shape[0] * TR.shape[0]) <= dcut):
        return merge_x(TL, TR, True)
    
    # Compute the merged tensor
    Tmerge_pure = np.einsum('abcd,ecfg,hbid,jifg->aehj', TL, TR, TL, TR, optimize = OPTIMIZE)
    Tmerge_pure_shape = Tmerge_pure.shape
    Tmerge_pure = Tmerge_pure.reshape(
        Tmerge_pure_shape[0] * Tmerge_pure_shape[1], Tmerge_pure_shape[2] * Tmerge_pure_shape[3]
    )
    # print(Tmerge_pure.shape)
    # Perform SVD with rank truncation
    U, S, Vt = torch.svd(torch.tensor(Tmerge_pure))
    U = U[:,:dcut].numpy()
    U = U.reshape(Tmerge_pure_shape[0], Tmerge_pure_shape[1], dcut)
    
    # Reconstruct the truncated tensor
    Tmerge = np.einsum('abcd,ecfg,aeh,dgi->hbfi', TL, TR, U, U, optimize = OPTIMIZE)
    return Tmerge


In [41]:
%%time
dcut = 4

for i,temp in enumerate(Temp):
    T_bare = get_T_bare(1/temp)
    TL = T_bare
    for j in range(2,MaxL+1):
        TLx = merge_x_truncate(TL,TL,dcut)
        TL_Trace = merge_y(TLx,TLx,True,True)
        eigvals, eigvecs = eigsh(TL_Trace, k=2, which='LM', return_eigenvectors=True)
        W[j,i,:] = eigvals
        TL = merge_y_truncate(TLx,TLx,dcut)
        TL = TL/np.mean(np.abs(TL))
E = -np.log(W)
Corr_len_4 = 1/(E[:,:,-2]-E[:,:,-1])
np.allclose(Corr_len_4, Corr_len_4_cytnx)

CPU times: user 583 ms, sys: 0 ns, total: 583 ms
Wall time: 584 ms


<timed exec>:14: RuntimeWarning: divide by zero encountered in divide


True

In [42]:
%%time
dcut = 10

for i,temp in enumerate(Temp):
    T_bare = get_T_bare(1/temp)
    TL = T_bare
    for j in range(2,MaxL+1):
        TLx = merge_x_truncate(TL,TL,dcut)
        TL_Trace = merge_y(TLx,TLx,True,True)
        eigvals, eigvecs = eigsh(TL_Trace, k=2, which='LM', return_eigenvectors=True)
        W[j,i,:] = eigvals
        TL = merge_y_truncate(TLx,TLx,dcut)
        TL = TL/np.mean(np.abs(TL))
E = -np.log(W)
Corr_len_10 = 1/(E[:,:,-2]-E[:,:,-1])
np.allclose(Corr_len_10, Corr_len_10_cytnx)

KeyboardInterrupt: 

In [9]:
np.allclose(Corr_len_10, Corr_len_10_cytnx)

NameError: name 'Corr_len_10' is not defined